In [1]:
# ==========================================
# Import Required Libraries
# ==========================================

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor

import warnings
warnings.filterwarnings("ignore")

print("✅ All libraries imported successfully")

✅ All libraries imported successfully


In [3]:
# ==========================================
# Load Final Cleaned Dataset
# ==========================================

file_path = r"C:\Users\naman\Desktop\train-delay-prediction\data\processed\final_cleaned_dataset.csv"

df = pd.read_csv(file_path)

print("✅ Dataset loaded")
print("Shape:", df.shape)
df.head()

✅ Dataset loaded
Shape: (7395517, 13)


,date,station_no,station_name,delay,train_no,year,month,day,day_of_week,total_route_distance,total_stops,type_code,station_zone
0,2025-04-14,20,GPC,85.0,19014,2025,4,14,0,643.0,61.0,EXP-TRAINS,WCR
1,2025-11-05,6,MGLP,14.0,54033,2025,11,5,2,161.0,29.0,PASS-TRAINS,NR
2,2026-01-19,56,KMC,64.0,17662,2026,1,19,0,602.0,81.0,EXP-TRAINS,SCR
3,2025-02-11,6,SRTL,19.0,56312,2025,2,11,1,57.0,12.0,PASS-TRAINS,SR
4,2025-09-15,4,IGP,38.0,12131,2025,9,15,0,336.0,8.0,SF-TRAINS,CR


In [4]:
# ==========================================
# Add Historical Aggregate Features
# ==========================================

# Train-level average delay
train_avg = df.groupby('train_no')['delay'].mean().reset_index()
train_avg.columns = ['train_no', 'train_avg_delay']

# Station-level average delay
station_avg = df.groupby('station_name')['delay'].mean().reset_index()
station_avg.columns = ['station_name', 'station_avg_delay']

# Merge into main dataset
df = df.merge(train_avg, on='train_no', how='left')
df = df.merge(station_avg, on='station_name', how='left')

print("✅ Historical features added")
df.head()

✅ Historical features added


,date,station_no,station_name,delay,train_no,year,month,day,day_of_week,total_route_distance,total_stops,type_code,station_zone,train_avg_delay,station_avg_delay
0,2025-04-14,20,GPC,85.0,19014,2025,4,14,0,643.0,61.0,EXP-TRAINS,WCR,83.764884,116.033097
1,2025-11-05,6,MGLP,14.0,54033,2025,11,5,2,161.0,29.0,PASS-TRAINS,NR,43.622233,37.479440
2,2026-01-19,56,KMC,64.0,17662,2026,1,19,0,602.0,81.0,EXP-TRAINS,SCR,42.675064,55.032166
3,2025-02-11,6,SRTL,19.0,56312,2025,2,11,1,57.0,12.0,PASS-TRAINS,SR,17.639440,35.369183
4,2025-09-15,4,IGP,38.0,12131,2025,9,15,0,336.0,8.0,SF-TRAINS,CR,17.863071,43.506216


In [5]:
# ==========================================
# Add Season Feature
# ==========================================

def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Summer'
    elif month in [6, 7, 8, 9]:
        return 'Monsoon'
    else:
        return 'Post-Monsoon'

df['season'] = df['month'].apply(get_season)

print("✅ Season feature added")
df['season'].value_counts()

✅ Season feature added


season
Monsoon         2502376
Summer          1846726
Winter          1785369
Post-Monsoon    1261046
Name: count, dtype: int64

In [6]:
# ==========================================
# Encode Categorical Columns using LabelEncoder
# ==========================================

categorical_cols = ['type_code', 'station_zone', 'season']

encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

print("✅ Categorical encoding complete")
df.head()

✅ Categorical encoding complete


,date,station_no,station_name,delay,train_no,year,month,day,day_of_week,total_route_distance,total_stops,type_code,station_zone,train_avg_delay,station_avg_delay,season
0,2025-04-14,20,GPC,85.0,19014,2025,4,14,0,643.0,61.0,0,17,83.764884,116.033097,2
1,2025-11-05,6,MGLP,14.0,54033,2025,11,5,2,161.0,29.0,2,8,43.622233,37.479440,1
2,2026-01-19,56,KMC,64.0,17662,2026,1,19,0,602.0,81.0,0,11,42.675064,55.032166,3
3,2025-02-11,6,SRTL,19.0,56312,2025,2,11,1,57.0,12.0,2,14,17.639440,35.369183,3
4,2025-09-15,4,IGP,38.0,12131,2025,9,15,0,336.0,8.0,5,0,17.863071,43.506216,0


In [7]:
# ==========================================
# Define Final Feature Set (X) and Target (y)
# ==========================================

feature_cols = [
    'month',
    'day',
    'day_of_week',
    'total_route_distance',
    'total_stops',
    'train_avg_delay',
    'station_avg_delay',
    'type_code',
    'station_zone',
    'season'
]

X = df[feature_cols]
y = df['delay']

print("✅ Feature matrix ready")
print("X shape:", X.shape)
print("y shape:", y.shape)

✅ Feature matrix ready
X shape: (7395517, 10)
y shape: (7395517,)


In [8]:
# ==========================================
# Train / Test Split (80/20)
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples :", X_test.shape[0])

Training samples: 5916413
Testing samples : 1479104


In [9]:
# ==========================================
# Train XGBoost Regressor Model
# ==========================================

model = XGBRegressor(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    tree_method="hist"   # Fast training for large data
)

model.fit(X_train, y_train)

print("✅ Model training complete!")

✅ Model training complete!


In [10]:
# ==========================================
# Evaluate Model Performance
# ==========================================

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("📊 Model Performance:")
print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²  :", round(r2, 4))

📊 Model Performance:
MAE : 24.43
RMSE: 44.65
R²  : 0.4322


In [11]:
train_station_avg = df.groupby(['train_no','station_name'])['delay'].mean().reset_index()
train_station_avg.columns = ['train_no','station_name','train_station_avg_delay']

In [12]:
zone_avg = df.groupby('station_zone')['delay'].mean().reset_index()
zone_avg.columns = ['station_zone', 'zone_avg_delay']

In [13]:
# ==========================================
# Add Powerful Aggregate Features
# ==========================================

# Feature 1: Train + Station combined avg delay
train_station_avg = df.groupby(['train_no', 'station_name'])['delay'].mean().reset_index()
train_station_avg.columns = ['train_no', 'station_name', 'train_station_avg_delay']

df = df.merge(train_station_avg, on=['train_no', 'station_name'], how='left')

# Feature 2: Zone-wise avg delay
zone_avg = df.groupby('station_zone')['delay'].mean().reset_index()
zone_avg.columns = ['station_zone', 'zone_avg_delay']

df = df.merge(zone_avg, on='station_zone', how='left')

print("✅ Powerful features added")
df.head()

✅ Powerful features added


,date,station_no,station_name,delay,train_no,year,month,day,day_of_week,total_route_distance,total_stops,type_code,station_zone,train_avg_delay,station_avg_delay,season,train_station_avg_delay,zone_avg_delay
0,2025-04-14,20,GPC,85.0,19014,2025,4,14,0,643.0,61.0,0,17,83.764884,116.033097,2,94.333333,44.111693
1,2025-11-05,6,MGLP,14.0,54033,2025,11,5,2,161.0,29.0,2,8,43.622233,37.479440,1,24.616667,34.628571
2,2026-01-19,56,KMC,64.0,17662,2026,1,19,0,602.0,81.0,0,11,42.675064,55.032166,3,82.633803,40.321691
3,2025-02-11,6,SRTL,19.0,56312,2025,2,11,1,57.0,12.0,2,14,17.639440,35.369183,3,23.347222,15.489922
4,2025-09-15,4,IGP,38.0,12131,2025,9,15,0,336.0,8.0,5,0,17.863071,43.506216,0,23.562500,36.371388


In [14]:
feature_cols = [
    'month',
    'day',
    'day_of_week',
    'total_route_distance',
    'total_stops',
    'train_avg_delay',
    'station_avg_delay',
    'train_station_avg_delay',   # NEW
    'zone_avg_delay',            # NEW
    'type_code',
    'station_zone',
    'season'
]

X = df[feature_cols]
y = df['delay']

print("New X shape:", X.shape)

New X shape: (7395517, 12)


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = XGBRegressor(
    n_estimators=400,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("📊 Updated Model Performance:")
print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²  :", round(r2, 4))

📊 Updated Model Performance:
MAE : 21.02
RMSE: 41.1
R²  : 0.5189


In [17]:
import os
print("Current working directory:", os.getcwd())

Current working directory: c:\Users\naman\Desktop\train-delay-prediction\notebooks\notebooks


In [18]:
# ==========================================
# Merge Distance From Origin (Safe Path)
# ==========================================

import os

# Auto-detect project root
BASE_DIR = r"C:\Users\naman\Desktop\train-delay-prediction"
schedule_path = os.path.join(BASE_DIR, "data", "raw", "indian_delay", "combined_schedule.csv")

schedule_df = pd.read_csv(schedule_path)

distance_df = schedule_df[['train_no', 'station_no', 'distance_from_origin']]

df = df.merge(distance_df, on=['train_no', 'station_no'], how='left')

print("✅ Distance from origin merged")
print("Missing values:", df['distance_from_origin'].isna().sum())

✅ Distance from origin merged
Missing values: 195498


In [19]:
# ==========================================
# Handle Missing Distance Values
# ==========================================

df['distance_from_origin'] = df['distance_from_origin'].fillna(
    df['distance_from_origin'].median()
)

print("✅ Missing distances handled")
print("Remaining missing:", df['distance_from_origin'].isna().sum())

✅ Missing distances handled
Remaining missing: 0


In [20]:
# ==========================================
# Add Late Ratio Feature per Train
# ==========================================

# Binary flag: was the train late (>5 min)?
df['is_late'] = (df['delay'] > 5).astype(int)

# Ratio: how often is this train late?
late_ratio = df.groupby('train_no')['is_late'].mean().reset_index()
late_ratio.columns = ['train_no', 'late_ratio']

# Merge into main dataframe
df = df.merge(late_ratio, on='train_no', how='left')

print("✅ Late ratio feature added")
df[['train_no', 'late_ratio']].head()

✅ Late ratio feature added


,train_no,late_ratio
0,19014,0.932791
1,54033,0.899899
2,17662,0.868085
3,56312,0.760793
4,12131,0.721992


In [21]:
# ==========================================
# Updated Feature Set for Training
# ==========================================

feature_cols = [
    'month',
    'day',
    'day_of_week',
    'station_no',
    'distance_from_origin',
    'total_route_distance',
    'total_stops',
    'train_avg_delay',
    'station_avg_delay',
    'train_station_avg_delay',
    'zone_avg_delay',
    'late_ratio',
    'type_code',
    'station_zone',
    'season'
]

X = df[feature_cols]
y = df['delay']

print("✅ New feature matrix ready")
print("X shape:", X.shape)
print("y shape:", y.shape)

✅ New feature matrix ready
X shape: (7395517, 15)
y shape: (7395517,)


In [22]:
# ==========================================
# Train Improved XGBoost Model (v3)
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = XGBRegressor(
    n_estimators=500,
    max_depth=9,
    learning_rate=0.08,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

model.fit(X_train, y_train)

print("✅ Model v3 training complete!")

✅ Model v3 training complete!


In [23]:
# ==========================================
# Evaluate Model v3
# ==========================================

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("📊 Version 3 Model Performance:")
print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²  :", round(r2, 4))

📊 Version 3 Model Performance:
MAE : 20.57
RMSE: 40.05
R²  : 0.5431


In [24]:
# ==========================================
# Save Trained XGBoost Model
# ==========================================

import joblib
import os

# Create model directory if not exists
os.makedirs("../model", exist_ok=True)

# Save the trained model
model_path = "../model/xgb_model.pkl"
joblib.dump(model, model_path)

print("✅ Model saved successfully at:", model_path)

✅ Model saved successfully at: ../model/xgb_model.pkl


In [29]:
# ==========================================
# Save Label Encoders
# ==========================================

encoder_path = "../model/encoders.pkl"
joblib.dump(encoders, encoder_path)

print("✅ Encoders saved at:", encoder_path)

✅ Encoders saved at: ../model/encoders.pkl


In [26]:
# ==========================================
# Save Feature Column List
# ==========================================

features_path = "../model/features.pkl"
joblib.dump(feature_cols, features_path)

print("✅ Feature list saved at:", features_path)

✅ Feature list saved at: ../model/features.pkl


In [27]:
# ==========================================
# Save Aggregation Lookup Tables
# ==========================================

lookup_data = {
    "train_avg": train_avg,
    "station_avg": station_avg,
    "train_station_avg": train_station_avg,
    "zone_avg": zone_avg,
    "late_ratio": late_ratio,
    "route_features": df[['train_no', 'total_route_distance', 'total_stops']].drop_duplicates()
}

lookup_path = "../model/lookup_tables.pkl"
joblib.dump(lookup_data, lookup_path)

print("✅ Lookup tables saved at:", lookup_path)

✅ Lookup tables saved at: ../model/lookup_tables.pkl


In [28]:
# ==========================================
# Quick Prediction Test
# ==========================================

sample = X_test.iloc[0:1]
prediction = model.predict(sample)

print("Predicted Delay:", round(prediction[0], 2), "minutes")
print("Actual Delay   :", y_test.iloc[0], "minutes")

Predicted Delay: 27.34 minutes
Actual Delay   : 30.0 minutes


In [32]:
# ==========================================
# Create ENHANCED Smart Lookup for Dashboard
# ==========================================

import os
import joblib
import pandas as pd

BASE_DIR = r"C:\Users\naman\Desktop\train-delay-prediction"

# Load raw datasets
station_info = pd.read_csv(
    os.path.join(BASE_DIR, "data", "raw", "indian_delay", "station_full_names.csv")
)
train_info = pd.read_csv(
    os.path.join(BASE_DIR, "data", "raw", "indian_delay", "train_details.csv")
)
schedule_info = pd.read_csv(
    os.path.join(BASE_DIR, "data", "raw", "indian_delay", "combined_schedule.csv")
)

# ---------- 1. Station code → full name map ----------
station_name_map = dict(
    zip(station_info['station_name'], station_info['station_full_name'])
)

# ---------- 2. Station display ----------
station_info['display_name'] = (
    station_info['station_full_name'] + " (" +
    station_info['station_name'] + ") - " +
    station_info['station_zone']
)

# ---------- 3. Train display with origin → destination ----------
def get_route_info(train_no):
    stops = schedule_info[
        schedule_info['train_no'] == train_no
    ].sort_values('station_no')

    if len(stops) == 0:
        return "Unknown", "Unknown"

    origin = station_name_map.get(
        stops.iloc[0]['station_name'], stops.iloc[0]['station_name']
    )
    dest = station_name_map.get(
        stops.iloc[-1]['station_name'], stops.iloc[-1]['station_name']
    )
    return origin, dest

train_display_list = []
for _, row in train_info.iterrows():
    origin, dest = get_route_info(row['train_no'])
    display = f"{row['train_no']} - {row['train_name']} ({origin} → {dest})"
    train_display_list.append(display)

train_info['display_name'] = train_display_list

# ---------- 4. Route lookup with full station names ----------
def build_route(train_no):
    stops = schedule_info[
        schedule_info['train_no'] == train_no
    ].sort_values('station_no')

    result = []
    for _, s in stops.iterrows():
        code = s['station_name']
        full_name = station_name_map.get(code, code)
        result.append({
            'station_no': s['station_no'],
            'station_name': code,
            'station_full_name': full_name
        })
    return result

route_lookup = {}
for train_no in schedule_info['train_no'].unique():
    route_lookup[train_no] = build_route(train_no)

# ---------- Save enhanced lookup ----------
smart_lookup = {
    "station_info": station_info,
    "train_info": train_info,
    "route_lookup": route_lookup,
    "station_name_map": station_name_map
}

joblib.dump(smart_lookup, os.path.join(BASE_DIR, "model", "smart_lookup.pkl"))

print("✅ Enhanced smart lookup saved!")
print(f"Total stations: {len(station_info)}")
print(f"Total trains: {len(train_info)}")
print(f"Total routes: {len(route_lookup)}")

# Sample check
print("\n🔍 Sample Rajdhani trains:")
sample = train_info[
    train_info['train_name'].str.contains('RAJDHANI', na=False)
]
print(sample['display_name'].head(5).to_string(index=False))

✅ Enhanced smart lookup saved!
Total stations: 8963
Total trains: 8992
Total routes: 8673

🔍 Sample Rajdhani trains:
12309 - RJPB RAJDHANI (RJNDR NGR BIHAR → NEW DE...
12310 - RJPB RAJDHANI (NEW DELHI → RJNDR NGR BI...
        12313 - RAJDHANI EXP (SEALDAH → NEW DELHI)
12423 - DBRT NDLS RAJDHANI (DIBRUGARH → NEW DELHI)
     12424 - DBRT RAJDHANI (NEW DELHI → DIBRUGARH)


In [33]:
# ==========================================
# Save Aggregated Data for Dashboard
# ==========================================

import os
import joblib

BASE_DIR = r"C:\Users\naman\Desktop\train-delay-prediction"

# Zone-wise average delay
zone_stats = df.groupby('station_zone').agg(
    avg_delay=('delay', 'mean'),
    total_records=('delay', 'count')
).reset_index()

# Reverse encoding for readability
zone_stats['station_zone'] = encoders['station_zone'].inverse_transform(
    zone_stats['station_zone'].astype(int)
)
zone_stats = zone_stats.sort_values('avg_delay', ascending=False)

# Monthly trend
monthly_stats = df.groupby('month').agg(
    avg_delay=('delay', 'mean'),
    total_records=('delay', 'count')
).reset_index()

# Train type analysis
type_stats = df.groupby('type_code').agg(
    avg_delay=('delay', 'mean'),
    total_records=('delay', 'count')
).reset_index()

type_stats['type_code'] = encoders['type_code'].inverse_transform(
    type_stats['type_code'].astype(int)
)
type_stats = type_stats.sort_values('avg_delay', ascending=False)

# Top 10 most delayed trains
top_delayed = df.groupby('train_no').agg(
    avg_delay=('delay', 'mean'),
    total_records=('delay', 'count')
).reset_index()

# Filter trains with at least 100 records (statistical significance)
top_delayed = top_delayed[top_delayed['total_records'] >= 100]
top_delayed = top_delayed.sort_values('avg_delay', ascending=False).head(10)

# Add train names to top delayed
train_names = pd.read_csv(
    os.path.join(BASE_DIR, "data", "raw", "indian_delay", "train_details.csv")
)
top_delayed = top_delayed.merge(
    train_names[['train_no', 'train_name', 'type_code']],
    on='train_no', how='left'
)

# Overall KPIs
kpis = {
    "total_records": len(df),
    "unique_trains": df['train_no'].nunique(),
    "unique_stations": df['station_name'].nunique(),
    "avg_delay_overall": df['delay'].mean(),
    "worst_zone": zone_stats.iloc[0]['station_zone'],
    "worst_zone_delay": zone_stats.iloc[0]['avg_delay']
}

# Save all
dashboard_data = {
    "zone_stats": zone_stats,
    "monthly_stats": monthly_stats,
    "type_stats": type_stats,
    "top_delayed": top_delayed,
    "kpis": kpis
}

joblib.dump(
    dashboard_data,
    os.path.join(BASE_DIR, "model", "dashboard_data.pkl")
)

print("✅ Dashboard aggregated data saved!")
print("\n📊 Preview:")
print("Total records:", kpis["total_records"])
print("Avg delay:", round(kpis["avg_delay_overall"], 2), "min")
print("Worst zone:", kpis["worst_zone"], "-", round(kpis["worst_zone_delay"], 2), "min")

✅ Dashboard aggregated data saved!

📊 Preview:
Total records: 7395517
Avg delay: 34.58 min
Worst zone: SCOR - 163.15 min


In [34]:
# ==========================================
# Add Zone Full Names Dictionary
# ==========================================

zone_full_names = {
    'NR': 'Northern Railway',
    'SR': 'Southern Railway',
    'ER': 'Eastern Railway',
    'WR': 'Western Railway',
    'CR': 'Central Railway',
    'SCR': 'South Central Railway',
    'SCoR': 'South Coast Railway',
    'SER': 'South Eastern Railway',
    'NER': 'North Eastern Railway',
    'NCR': 'North Central Railway',
    'WCR': 'West Central Railway',
    'ECR': 'East Central Railway',
    'SWR': 'South Western Railway',
    'NWR': 'North Western Railway',
    'ECoR': 'East Coast Railway',
    'SECR': 'South East Central Railway',
    'NFR': 'Northeast Frontier Railway',
    'KR': 'Konkan Railway',
    'Unknown': 'Unknown Zone'
}

# Update dashboard data with full names
import joblib
import os

BASE_DIR = r"C:\Users\naman\Desktop\train-delay-prediction"

# Load existing dashboard data
dashboard_data = joblib.load(os.path.join(BASE_DIR, "model", "dashboard_data.pkl"))

# Add zone full names
dashboard_data['zone_full_names'] = zone_full_names

# Update zone_stats with full names
zone_stats = dashboard_data['zone_stats']
zone_stats['zone_full_name'] = zone_stats['station_zone'].map(zone_full_names).fillna(zone_stats['station_zone'])
dashboard_data['zone_stats'] = zone_stats

# Update KPI worst zone with full name
worst_zone_code = dashboard_data['kpis']['worst_zone']
dashboard_data['kpis']['worst_zone_full'] = zone_full_names.get(worst_zone_code, worst_zone_code)

# Save updated
joblib.dump(dashboard_data, os.path.join(BASE_DIR, "model", "dashboard_data.pkl"))

print("✅ Zone full names added to dashboard data")
print("\nSample zones:")
print(zone_stats[['station_zone', 'zone_full_name', 'avg_delay']].head(10))

✅ Zone full names added to dashboard data

Sample zones:
   station_zone              zone_full_name   avg_delay
10         SCOR                        SCOR  163.153846
12         SECR  South East Central Railway   55.801731
2           ECR        East Central Railway   51.360906
1          ECOR                        ECOR   49.436356
13          SER       South Eastern Railway   49.098252
4            KR              Konkan Railway   45.225987
5           NCR       North Central Railway   44.460050
6           NER       North Eastern Railway   44.344489
17          WCR        West Central Railway   44.111693
11          SCR       South Central Railway   40.321691


In [39]:
# ==========================================
# Test City Name Cleaning (Standalone)
# ==========================================

def clean_city_name(station_full_name: str) -> str:
    """Extract proper city name from railway station name."""
    if not station_full_name:
        return "Delhi"
    
    name = str(station_full_name).strip().upper()
    
    prefixes_to_remove = [
        'V ', 'NEW ', 'OLD ', 'PT ', 'SMT ', 'DR ', 'SR ',
        'HAZRAT ', 'MAHAVEER ', 'SHRI ', 'SHREE ',
        'CHHATRAPATI SHIVAJI MAHARAJ ', 'CHHATRAPATI '
    ]
    
    suffixes_to_remove = [
        ' JN', ' JUNCTION', ' CANTT', ' CANTONMENT',
        ' TERMINUS', ' TERMINAL', ' CENTRAL',
        ' HALT', ' CITY', ' STATION',
        ' ROAD', ' RD', ' TMS', ' TERM'
    ]
    
    # Remove prefixes
    changed = True
    while changed:
        changed = False
        for prefix in prefixes_to_remove:
            if name.startswith(prefix):
                name = name[len(prefix):].strip()
                changed = True
                break
    
    # Remove suffixes
    changed = True
    while changed:
        changed = False
        for suffix in suffixes_to_remove:
            if name.endswith(suffix):
                name = name[:-len(suffix)].strip()
                changed = True
                break
    
    words = name.split()
    if len(words) > 2:
        name = words[-1]
    elif len(words) == 2:
        name = words[-1]
    
    return name.title() if name else "Delhi"


# Test cases
test_names = [
    "V LAKSHMIBAI JHANSI",
    "NEW DELHI",
    "H NIZAMUDDIN",
    "MUMBAI CENTRAL",
    "HOWRAH JN",
    "LUCKNOW",
    "BINA JN",
    "GWALIOR",
    "LALITPUR",
    "MORENA"
]

print("City Name Cleaning Test:\n")
for name in test_names:
    cleaned = clean_city_name(name)
    print(f"{name:35s} → {cleaned}")

City Name Cleaning Test:

V LAKSHMIBAI JHANSI                 → Jhansi
NEW DELHI                           → Delhi
H NIZAMUDDIN                        → Nizamuddin
MUMBAI CENTRAL                      → Mumbai
HOWRAH JN                           → Howrah
LUCKNOW                             → Lucknow
BINA JN                             → Bina
GWALIOR                             → Gwalior
LALITPUR                            → Lalitpur
MORENA                              → Morena


In [1]:
import requests
import os
from dotenv import load_dotenv

load_dotenv(r"C:\Users\naman\Desktop\train-delay-prediction\.env")
api_key = os.getenv("RAILRADAR_API_KEY", "")

# Test Malwa Express
url = "https://api.railradar.in/v1/trains/11055/live"
headers = {"Authorization": f"Bearer {api_key}"}

response = requests.get(url, headers=headers, timeout=10)
data = response.json()

if data.get("success"):
    d = data["data"]
    print("Status:", d.get("status"))
    print("Is Live:", d.get("isLive"))
    print("Current Delay:", d.get("delayMinutes"))
    print("Last Updated:", d.get("lastUpdatedAt"))
    print("Current Location:", d.get("currentLocation", {}).get("stationCode"))
    print("Current Sequence:", d.get("currentLocation", {}).get("sequence"))

Status: running
Is Live: True
Current Delay: 6
Last Updated: 2026-07-04T14:27:58+05:30
Current Location: KS
Current Sequence: 219
